In [4]:
import logging
import sys
from collections.abc import Mapping
from pathlib import Path
from typing import Any

from pydantic import BaseModel, ConfigDict

from wags_llm.client.bedrock import BedrockClaudeJsonClient, EffortLevel
from wags_llm.registry.base import Registry
from wags_llm.services.structured_task import StructuredTaskRunner
from wags_llm.templates.skill_template import SkillTemplate

logging.basicConfig(
    stream=sys.stdout,
    level=logging.WARNING,
    format="%(name)s - %(levelname)s - %(message)s",
)
logging.getLogger("wags_llm").setLevel(logging.DEBUG)

# Effort Demo
Set effort parameter to do some prompt task. Description description

In [5]:
class VariantCurationSkill(SkillTemplate):
    skill_path = Path("skills/variant_curation_0.1.0.md")

    def build_user_prompt(self, payload: Mapping[str, Any]) -> str:
        variant = payload["variant"]
        disease = payload.get("disease", "cancer")

        return f"""Curate the following variant for {disease}.
Variant:
{variant}

Return concise JSON matching the provided schema:
- clinical_significance: one short sentence
- evidence_level: short label
- supporting_rationale: 2-3 short sentences maximum
"""


class VariantCurationResult(BaseModel):
    model_config = ConfigDict(extra="forbid", use_enum_values=True)  # Required

    clinical_significance: str | None = None
    evidence_level: str | None = None
    supporting_rationale: str | None = None
    error_message: str | None = None

skill = VariantCurationSkill()
skill.name, skill.version

registry = Registry()
registry.register(skill)

wags_llm.registry.base - DEBUG - Registering template: name='variant_curation', version='0.1.0', template_type='skill'


In [6]:
# LOW
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 350

llm_client = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=MAX_TOKENS,
    temperature=1,
    effort=EffortLevel.LOW # LOW, MEDIUM, HIGH, MAX
)

task_runner = StructuredTaskRunner(
    client=llm_client,
    registry=registry,
)

result1 = task_runner.execute_skill(
    skill_name="variant_curation",
    skill_version="0.1.0",
    payload={
        "variant": "BRCA2 p.K3326*",
        "disease": "hereditary breast cancer",
    },

    response_model=VariantCurationResult,
)
result1

wags_llm.client.bedrock - DEBUG - BedrockClaudeJsonClient config: model_id='us.anthropic.claude-sonnet-4-6', region_name='us-east-1', profile_name='dev-account', max_tokens=350, temperature=1.000000, effort=low
wags_llm.client.bedrock - INFO - BedrockClaudeJsonClient successfully initialized for model_id='us.anthropic.claude-sonnet-4-6'
wags_llm.templates.skill_template - DEBUG - Loading skill from path: skills/variant_curation_0.1.0.md
wags_llm.templates.skill_template - INFO - Loaded skill from path: skills/variant_curation_0.1.0.md
wags_llm.client.bedrock - DEBUG - Bedrock Claude usage={'inputTokens': 583, 'outputTokens': 176, 'totalTokens': 759, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
wags_llm.client.bedrock - DEBUG - Bedrock Claude metrics={'latencyMs': 19134}
wags_llm.client.bedrock - DEBUG - Bedrock Claude content=[{'text': '{"clinical_significance":"BRCA2 p.K3326* is considered a benign/likely benign variant not causative of hereditary breast and ovarian cancer s

VariantCurationResult(clinical_significance='BRCA2 p.K3326* is considered a benign/likely benign variant not causative of hereditary breast and ovarian cancer syndrome.', evidence_level='Benign – Strong (ClinVar/population evidence)', supporting_rationale='This truncating variant removes only the final 93 amino acids of BRCA2 and does not disrupt the critical DNA-binding or RAD51-interaction domains. Population studies show it occurs at relatively high frequency (MAF ~1%) in general populations, inconsistent with a high-penetrance pathogenic variant. Large case-control studies and functional data support its classification as benign, though some data suggest a modest modifying effect on cancer risk that is not clinically actionable.', error_message=None)

In [7]:
# MEDIUM
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 350

llm_client = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=MAX_TOKENS,
    temperature=1,
    effort=EffortLevel.MEDIUM # LOW, MEDIUM, HIGH, MAX
)

task_runner = StructuredTaskRunner(
    client=llm_client,
    registry=registry,
)

result2 = task_runner.execute_skill(
    skill_name="variant_curation",
    skill_version="0.1.0",
    payload={
        "variant": "BRCA2 p.K3326*",
        "disease": "hereditary breast cancer",
    },
    response_model=VariantCurationResult,
)
result2

wags_llm.client.bedrock - DEBUG - BedrockClaudeJsonClient config: model_id='us.anthropic.claude-sonnet-4-6', region_name='us-east-1', profile_name='dev-account', max_tokens=350, temperature=1.000000, effort=medium
wags_llm.client.bedrock - INFO - BedrockClaudeJsonClient successfully initialized for model_id='us.anthropic.claude-sonnet-4-6'
wags_llm.templates.skill_template - DEBUG - Loading skill from path: skills/variant_curation_0.1.0.md
wags_llm.templates.skill_template - INFO - Loaded skill from path: skills/variant_curation_0.1.0.md
wags_llm.client.bedrock - DEBUG - Bedrock Claude usage={'inputTokens': 583, 'outputTokens': 350, 'totalTokens': 933, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
wags_llm.client.bedrock - DEBUG - Bedrock Claude metrics={'latencyMs': 26638}
wags_llm.client.bedrock - DEBUG - Bedrock Claude content=[{'reasoningContent': {'reasoningText': {'text': "BRCA2 p.K3326* is a well-known variant. It's a truncating variant near the end of the BRCA2 protein

RuntimeError: skill execution failed for variant_curation version 0.1.0: Model returned non-JSON output: Unterminated string starting at: line 1 column 231 (char 230); output='{"clinical_significance":"BRCA2 p.K3326* is classified as Benign/Likely Benign for hereditary breast and ovarian cancer risk.","evidence_level":"B/LB (ClinVar consensus; CIViC-equivalent Level C–D population/functional evidence)","supporting'

In [8]:
# HIGH
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 350

llm_client = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=MAX_TOKENS,
    temperature=1,
    effort=EffortLevel.HIGH # LOW, MEDIUM, HIGH, MAX
)

task_runner = StructuredTaskRunner(
    client=llm_client,
    registry=registry,
)

result3 = task_runner.execute_skill(
    skill_name="variant_curation",
    skill_version="0.1.0",
    payload={
        "variant": "BRCA2 p.K3326*",
        "disease": "hereditary breast cancer",
    },
    response_model=VariantCurationResult,
)
result3

wags_llm.client.bedrock - DEBUG - BedrockClaudeJsonClient config: model_id='us.anthropic.claude-sonnet-4-6', region_name='us-east-1', profile_name='dev-account', max_tokens=350, temperature=1.000000, effort=high
wags_llm.client.bedrock - INFO - BedrockClaudeJsonClient successfully initialized for model_id='us.anthropic.claude-sonnet-4-6'
wags_llm.templates.skill_template - DEBUG - Loading skill from path: skills/variant_curation_0.1.0.md
wags_llm.templates.skill_template - INFO - Loaded skill from path: skills/variant_curation_0.1.0.md
wags_llm.client.bedrock - DEBUG - Bedrock Claude usage={'inputTokens': 583, 'outputTokens': 350, 'totalTokens': 933, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
wags_llm.client.bedrock - DEBUG - Bedrock Claude metrics={'latencyMs': 8286}
wags_llm.client.bedrock - DEBUG - Bedrock Claude content=[{'reasoningContent': {'reasoningText': {'text': "The user wants me to curate BRCA2 p.K3326* for hereditary breast cancer using CIViC-style evidence rea

RuntimeError: skill execution failed for variant_curation version 0.1.0: No text content found in Converse response: {'ResponseMetadata': {'RequestId': 'c31f3ba5-6b97-44b1-bd45-570dd95e58fc', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 06 Jul 2026 16:56:41 GMT', 'content-type': 'application/json', 'content-length': '2976', 'connection': 'keep-alive', 'x-amzn-requestid': 'c31f3ba5-6b97-44b1-bd45-570dd95e58fc'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'reasoningContent': {'reasoningText': {'text': "The user wants me to curate BRCA2 p.K3326* for hereditary breast cancer using CIViC-style evidence reasoning.\n\nBRCA2 p.K3326* (also written as K3326X or c.9976A>T) is a well-known variant. Let me recall what I know about it:\n\n1. BRCA2 p.K3326* is a truncating variant (nonsense mutation) located near the very end of the BRCA2 protein (the last exon, exon 27).\n2. This variant has been extensively studied and is classified as a **polymorphism/be

In [9]:
# MAX
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 350

llm_client = BedrockClaudeJsonClient(
    model_id=MODEL_ID,
    region_name=REGION_NAME,
    profile_name=PROFILE_NAME,
    max_tokens=MAX_TOKENS,
    temperature=1,
    effort=EffortLevel.MAX # LOW, MEDIUM, HIGH
)

task_runner = StructuredTaskRunner(
    client=llm_client,
    registry=registry,
)

result4 = task_runner.execute_skill(
    skill_name="variant_curation",
    skill_version="0.1.0",
    payload={
        "variant": "BRCA2 p.K3326*",
        "disease": "hereditary breast cancer",
    },
    response_model=VariantCurationResult,
)
result4

wags_llm.client.bedrock - DEBUG - BedrockClaudeJsonClient config: model_id='us.anthropic.claude-sonnet-4-6', region_name='us-east-1', profile_name='dev-account', max_tokens=350, temperature=1.000000, effort=max
wags_llm.client.bedrock - INFO - BedrockClaudeJsonClient successfully initialized for model_id='us.anthropic.claude-sonnet-4-6'
wags_llm.templates.skill_template - DEBUG - Loading skill from path: skills/variant_curation_0.1.0.md
wags_llm.templates.skill_template - INFO - Loaded skill from path: skills/variant_curation_0.1.0.md
wags_llm.client.bedrock - DEBUG - Bedrock Claude usage={'inputTokens': 583, 'outputTokens': 350, 'totalTokens': 933, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
wags_llm.client.bedrock - DEBUG - Bedrock Claude metrics={'latencyMs': 7426}
wags_llm.client.bedrock - DEBUG - Bedrock Claude content=[{'reasoningContent': {'reasoningText': {'text': 'The user wants me to curate the BRCA2 p.K3326* variant in the context of hereditary breast cancer, foll

RuntimeError: skill execution failed for variant_curation version 0.1.0: No text content found in Converse response: {'ResponseMetadata': {'RequestId': '1b36dc62-153e-48d4-8699-55f3c861ccbb', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 06 Jul 2026 16:57:10 GMT', 'content-type': 'application/json', 'content-length': '2836', 'connection': 'keep-alive', 'x-amzn-requestid': '1b36dc62-153e-48d4-8699-55f3c861ccbb'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'reasoningContent': {'reasoningText': {'text': 'The user wants me to curate the BRCA2 p.K3326* variant in the context of hereditary breast cancer, following CIViC-style evidence reasoning.\n\nLet me think about what I know about BRCA2 p.K3326* (also written as K3326X or c.9976A>T, p.Lys3326Ter):\n\nBRCA2 p.K3326* is a truncating variant (nonsense mutation) that introduces a premature stop codon at position 3326 in the BRCA2 protein. This is near the C-terminus of the BRCA2 protein.\n\nKey point